In [1]:
# Imports
import torch
import torch.nn as nn
import torchvision

In [2]:
# Let's load the dataset
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

size = (64, 64)
transform = torchvision.transforms.Compose([torchvision.transforms.Resize(size), torchvision.transforms.ToTensor()])
train_dataset = list(torchvision.datasets.Flowers102("/tmp/flowers", "train", transform=transform, download=True))
test_dataset = list(torchvision.datasets.Flowers102("/tmp/flowers", "test", transform=transform, download=True))



In [3]:
train_images = torch.stack([img for img, _ in train_dataset], dim=0).to(device)
test_images = torch.stack([img for img, _ in test_dataset], dim=0).to(device)
train_labels = torch.tensor([label for _, label in train_dataset]).to(device)
test_labels = torch.tensor([label for _, label in test_dataset]).to(device)

# Let's make sure we only have two classes
train_images, train_labels = train_images[train_labels < 2], train_labels[train_labels < 2]
test_images, test_labels = test_images[test_labels < 2], test_labels[test_labels < 2]

In [ ]:
model = torch.nn.Linear(3 * 64 * 64, 1).to(device)
################THIS IS A CHANGE FROM LINEAR TO BINARY###################
loss = torch.nn.BCEWithLogitsLoss()

#test1
#optimizer = torch.optim.SGD(model.parameters(), lr=.0001, momentum=0)

#test2
#optimizer = torch.optim.SGD(model.parameters(), lr=.001, momentum=0)

#test3
optimizer = torch.optim.SGD(model.parameters(), lr=.01, momentum=0)

for epoch in range(100):
    #computes the output of the model (same as linear)
    out = model(train_images.view(-1, 3 * 64 * 64))

    #converts the output to a loss value (probability of being in class 1) aka binary regression
    loss_val = loss(out.squeeze(), train_labels.float())

    #compute gradients and update the weights
    optimizer.zero_grad()
    loss_val.backward()

    optimizer.step()
    print(f"{epoch=}: {loss_val.item()}")

    #results 1
    #learning rate .0001
    #starts just below .7 [which is 50/50 coin flip]
    #gets better to .5 after 100 epochs but that is still not good enough for a real model
    #could increase learning rate.

    #results 2
    #learning rate .001
    #starts just above .7 [which is worse than 50/50 coin flip]
    #gets better to .19 after 100 epochs 
    
    #results 3
    #learning rate .01
    #starts just above .7 [which is worse than 50/50 coin flip]
    #gets better to .027 after 100 epochs [worse than the .001 learning rate but still good enough for a real model]
    #note that it got better than worse then better then worse as epochs went on, which is a sign of overfitting. We could try to reduce the learning rate or add regularization to prevent overfitting.
    #epoch=0: 0.7102622985839844
    #epoch=1: 0.5374635457992554
    #epoch=2: 0.8411537408828735
    #epoch=3: 2.120211601257324
    #epoch=4: 0.8836720585823059
    #epoch=5: 1.5546934604644775

    #go back to results 2 and move towards accuracy compared to the test images and labels since the model above is based on the dataset


epoch=0: 0.7102622985839844
epoch=1: 0.5374635457992554
epoch=2: 0.8411537408828735
epoch=3: 2.120211601257324
epoch=4: 0.8836720585823059
epoch=5: 1.5546934604644775
epoch=6: 0.6308847069740295
epoch=7: 0.731556236743927
epoch=8: 0.41383785009384155
epoch=9: 0.31157931685447693
epoch=10: 0.1956137716770172
epoch=11: 0.14839191734790802
epoch=12: 0.12452485412359238
epoch=13: 0.11565669625997543
epoch=14: 0.11046084016561508
epoch=15: 0.10663274675607681
epoch=16: 0.10325109958648682
epoch=17: 0.10011106729507446
epoch=18: 0.0971546545624733
epoch=19: 0.09436038136482239
epoch=20: 0.09171471744775772
epoch=21: 0.08920620381832123
epoch=22: 0.0868249163031578
epoch=23: 0.0845615491271019
epoch=24: 0.08240790665149689
epoch=25: 0.08035637438297272
epoch=26: 0.07840006053447723
epoch=27: 0.07653268426656723
epoch=28: 0.07474850118160248
epoch=29: 0.07304223626852036
epoch=30: 0.07140897214412689
epoch=31: 0.0698443278670311
epoch=32: 0.0683441087603569
epoch=33: 0.06690452247858047
epoch=

In [ ]:
test_images_01 = test_images[test_labels < 2]
test_labels_01 = test_labels[test_labels < 2]

#predict test labels ( 0 or 1) based on the test images and the model we just trained
# negative means class 0 and positive means class 1
pred_test = model(test_images_01.view(-1, 3 * 64 * 64))
#print(pred_test)
# negative means class 0 and positive means class 1

################THIS IS A CHANGE FROM LINEAR TO BINARY to make it outputs of 0 or 1 and then compare with the true labels###################

#better display of correct and incorrect predictions than just printing the predicted values [print(pred_test)]
print(((pred_test[:, 0] > 0).int() == test_labels_01).float().mean())
#.7667 is pretty good because that is the chance of the model getting it right
#could be better if we trained for more epochs or used a different learning rate or added regularization to prevent overfitting.


tensor(0.7667)


In [ ]:
def accuracy(pred: torch.Tensor, label: torch.Tensor) -> float:
    return ((pred > 0.5) == label).float().mean().item()


model = torch.nn.Linear(in_features=size[0] * size[1] * 3, out_features=1)
model = model.to(device)

loss_fn = nn.BCEWithLogitsLoss()
optim = torch.optim.SGD(model.parameters(), lr=2e-2)
num_epochs = 500

for epoch in range(num_epochs):
    pred = model(train_images.view(train_images.shape[0], -1))[..., 0]
    loss_val = loss_fn(pred, train_labels.float())

    optim.zero_grad()
    loss_val.backward()
    optim.step()

    if epoch % 25 == 0 or epoch == num_epochs - 1:
        print(f"{epoch =:5d}  loss = {loss_val.item():.2f}  accuracy(train) = {accuracy(pred, train_labels):.3f}")

    if epoch % 100 == 0 or epoch == num_epochs - 1:
        with torch.inference_mode():
            pred = model(test_images.view(test_images.shape[0], -1))[..., 0]
            print(f"   Accuracy (test): {accuracy(pred, test_labels):.3f}")